In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

In [2]:
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

def finbert_sentiment(subject, body):
    text = subject + " " + body if body else subject
    text = text.strip()

    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    # Run model
    with torch.no_grad():
        outputs = model(**inputs)

    # Get probabilities
    probs = F.softmax(outputs.logits, dim=-1)

    # Labels: [0: positive, 1: negative, 2: neutral]
    labels = ["positive", "negative", "neutral"]
    pred = torch.argmax(probs, dim=1).item()

    return {
        "finbert_label": labels[pred],
        "finbert_sentiment": {labels[i]: probs[0][i].item() for i in range(len(labels))}
    }

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [4]:
# Example usage
subject = "Stocks rally as market rebounds"
body = "The stock market saw a significant rebound today with major indices closing higher."
result = finbert_sentiment(subject, body)
print(result)

{'finbert_label': 'positive', 'finbert_sentiment': {'positive': 0.9457743763923645, 'negative': 0.028774550184607506, 'neutral': 0.02545112930238247}}
